# Naval Vessel Detector — Cloud GPU Training (Colab)

Runtime > Change runtime type > **GPU** (T4 is fine to start; A100 if you have Colab Pro+).

Steps: clone/upload repo -> mount Drive for persistence -> install deps -> download+prep data -> train -> download weights.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Recommended: keep the repo + datasets + checkpoints on Drive so a Colab
# disconnect doesn't lose your progress. Adjust this path to where you
# uploaded/cloned naval-ai.
PROJECT_DIR = '/content/drive/MyDrive/naval-ai'

In [ ]:
%cd {PROJECT_DIR}/backend
!pip install -q ultralytics kaggle

## Dataset prep
Upload your `kaggle.json` (from kaggle.com/settings) to `~/.kaggle/kaggle.json` first if you want the Kaggle datasets pulled automatically.

In [ ]:
!mkdir -p ~/.kaggle && cp {PROJECT_DIR}/kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
%cd training
!python download_datasets.py --target ../../datasets/raw
# Read the printed manual-download instructions for HRSC2016/DOTA/Singapore
# Maritime — grab those separately and drop them in datasets/raw/<Name>/ before
# the next cell.

In [ ]:
!python prepare_dataset.py --raw-dir ../../datasets/raw --out-dir ../../datasets

## Weather augmentation (optional but recommended)
Synthesizes fog/rain/snow/night variants of the training split so the model sees weather diversity even though no single public dataset covers all 20 ship classes across all weather conditions.

In [ ]:
!python augment_weather.py --dataset-dir ../../datasets --variants-per-image 2

## Train
`--save-period 10` checkpoints every 10 epochs to Drive so a Colab disconnect doesn't cost you the whole run — resume with `--resume`.

In [ ]:
!python train.py --data ../../datasets/naval_dataset.yaml --epochs 150 --batch 16 --output-dir /content/drive/MyDrive/naval-ai/models/runs

## If disconnected mid-run, resume like this:

In [ ]:
# !python train.py --resume /content/drive/MyDrive/naval-ai/models/runs/naval_yolo/weights/last.pt

## Export ONNX (optional, for lighter-weight deployment)

In [ ]:
!python train.py --export-only --weights ../../models/weights/best.pt

`best.pt` is now at `models/weights/best.pt` in your Drive-backed project — that's exactly where `backend/inference/detector.py` looks for it, so the API picks it up with no further config.